# E-Commerce Sales Data — Exploratory Data Analysis

**ApexPlanet Data Analytics Internship — Task 1**

**Prepared by:** Sainath

This notebook performs environment setup verification, data understanding, data cleaning, statistical analysis, visualization, and insight generation.

## 1. Project Objective

The objective is to analyze e-commerce sales data to understand order distribution, sales and profit patterns, category performance, and relationships between numerical variables.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (8, 5)
PROJECT_ROOT = Path.cwd().parent
RAW_PATH = PROJECT_ROOT / 'data' / 'raw' / 'ecommerce_sales_raw.csv'
PROCESSED_PATH = PROJECT_ROOT / 'data' / 'processed' / 'cleaned_ecommerce_sales.csv'
print('Libraries imported successfully.')

## 2. Load the Dataset

The raw dataset is stored in `data/raw/`. It is a synthetic educational e-commerce dataset created for this internship practice project.

In [ ]:
df = pd.read_csv(RAW_PATH)
print('Shape:', df.shape)
df.head()

## 3. Data Understanding

In [ ]:
print('Rows and columns:', df.shape)
print('\nColumn names:')
print(df.columns.tolist())
print('\nData types and non-null counts:')
df.info()

In [ ]:
print('Missing values before cleaning:')
print(df.isnull().sum())
print('\nDuplicate rows before cleaning:', df.duplicated().sum())

In [ ]:
df.describe(include='all').T

## 4. Data Dictionary

| Column | Description |
|---|---|
| Order_ID | Unique order identifier |
| Order_Date | Date on which the order was placed |
| Customer_Name | Customer name; missing values are replaced during cleaning |
| Category | Product category |
| Product | Product purchased |
| Quantity | Number of units purchased |
| Sales | Net sales amount after discount |
| Profit | Profit earned from the order |
| City | Customer city |
| Payment_Method | Payment method used |

## 5. Data Cleaning and Preprocessing

Cleaning steps:
1. Convert date and numeric columns to suitable data types.
2. Fill missing text values with meaningful labels.
3. Fill missing numeric values using the median.
4. Remove duplicate records.
5. Save the cleaned dataset in `data/processed/`.

In [ ]:
cleaned = df.copy()

cleaned['Order_Date'] = pd.to_datetime(cleaned['Order_Date'], errors='coerce')
cleaned['Quantity'] = pd.to_numeric(cleaned['Quantity'], errors='coerce')
cleaned['Sales'] = pd.to_numeric(cleaned['Sales'], errors='coerce')
cleaned['Profit'] = pd.to_numeric(cleaned['Profit'], errors='coerce')

cleaned['Customer_Name'] = cleaned['Customer_Name'].fillna('Unknown Customer')
cleaned['City'] = cleaned['City'].fillna('Unknown City')
cleaned['Quantity'] = cleaned['Quantity'].fillna(cleaned['Quantity'].median()).round().astype(int)
cleaned['Sales'] = cleaned['Sales'].fillna(cleaned['Sales'].median())
cleaned['Profit'] = cleaned['Profit'].fillna(cleaned['Profit'].median())

before = len(cleaned)
cleaned = cleaned.drop_duplicates().reset_index(drop=True)
removed_duplicates = before - len(cleaned)

cleaned.to_csv(PROCESSED_PATH, index=False)
print('Duplicates removed:', removed_duplicates)
print('Cleaned shape:', cleaned.shape)
print('\nMissing values after cleaning:')
print(cleaned.isnull().sum())

## 6. Statistical Summary

In [ ]:
cleaned[['Quantity', 'Sales', 'Profit']].describe()

In [ ]:
print('Orders by category:')
print(cleaned['Category'].value_counts())

print('\nPayment method distribution:')
print(cleaned['Payment_Method'].value_counts())

## 7. Univariate Analysis

Univariate analysis studies one variable at a time.

In [ ]:
cleaned['Category'].value_counts().plot(kind='bar')
plt.title('Number of Orders by Category')
plt.xlabel('Category')
plt.ylabel('Number of Orders')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
sns.histplot(cleaned['Sales'], kde=True)
plt.title('Sales Distribution')
plt.xlabel('Sales')
plt.show()

In [ ]:
sns.boxplot(x=cleaned['Sales'])
plt.title('Sales Boxplot')
plt.xlabel('Sales')
plt.show()

## 8. Bivariate Analysis

Bivariate analysis studies the relationship between two variables.

In [ ]:
sns.scatterplot(data=cleaned, x='Sales', y='Profit', hue='Category')
plt.title('Sales vs Profit')
plt.show()

In [ ]:
numeric_cols = ['Quantity', 'Sales', 'Profit']
sns.heatmap(cleaned[numeric_cols].corr(), annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Correlation Matrix')
plt.show()

In [ ]:
category_summary = cleaned.groupby('Category').agg(
    Orders=('Order_ID', 'count'),
    Total_Sales=('Sales', 'sum'),
    Total_Profit=('Profit', 'sum'),
    Average_Sales=('Sales', 'mean')
).sort_values('Total_Sales', ascending=False)
category_summary

In [ ]:
city_summary = cleaned.groupby('City')['Sales'].sum().sort_values(ascending=False)
city_summary

## 9. Key Insights

Run the analysis cells above and replace or refine these points using the actual output:

- The category with the highest total sales can be identified from `category_summary`.
- The city with the highest sales can be identified from `city_summary`.
- The sales histogram shows the distribution of order values.
- The boxplot helps identify unusually high or low sales values for further investigation.
- The scatter plot helps examine the relationship between sales and profit.
- The correlation matrix shows the strength and direction of relationships among quantity, sales, and profit.

## 10. Conclusion

This project demonstrated the complete foundational analytics workflow: loading data, understanding its structure, cleaning data-quality issues, generating descriptive statistics, creating visualizations, and documenting business-oriented findings. The cleaned dataset is saved in `data/processed/cleaned_ecommerce_sales.csv`.